# Build design: W3123370480, attempt 02

## Retained scope and source interpretation

This attempt reproduces only the three retained Table 1, Panel B estimates for Eq. (1): one-, two-, and three-year-ahead earnings. It does not estimate four- or five-year horizons, the IBES comparison models, or any ICC valuation model.

The paper says pooled cross-sectional regressions are estimated each year from 1968 through 2008 using the prior ten years, and that coefficients are averaged across annual fits with Newey–West time-series inference (physical PDF pp. 4–6; printed pp. 507–509, Eq. (1), Table 1 Panel B and notes 10–11). It does not name the estimator. I use unweighted pooled OLS for each origin/horizon as a best-effort interpretation of the linear regression description. All six RHS variables and an intercept are retained. There are no firm or industry fixed effects, financial/utility exclusions, or firm-history minimums.

The fit year is the June information-origin year. An annual Compustat date ending April of year `t−1` through March of `t` is assigned to origin `t`, matching the paper's three-month reporting-lag description (physical p. 5; printed p. 508). For a horizon `τ`, a training outcome is the realized earnings observation in one of the ten preceding origin years, `t−10` through `t−1`; its predictor record is the same GVKEY's observation exactly `τ` origin years earlier. This makes every fitted outcome historical at `t`, uses dated fiscal records rather than row-position leads, and preserves gaps. The source does not say whether its ten-year window is defined on predictor-year or outcome-year, so the outcome-year convention is an explicit best-effort choice. FYE date distributions and the actual support by origin/horizon are recorded in the SAS listing.

The source states that Compustat annual data begin in 1963, although a literal ten-year outcome window for the 1968 origin starts in 1958 (physical pp. 3–4; printed pp. 506–507). Raw statements from 1957 onward are retained so the first usable balance-sheet accrual can have a prior fiscal-year record. Regression-period rows require Compustat `FYEAR>=1963` and June formation labels from 1963 onward. Thus the early fits use the available shortened window; no post-origin earnings are used to fill it. The actual start/end dates in local `CRSP.MSF` and the first June security observations are reported from the source file rather than inferred. Table 1 does not report fit-level Ns.

The source does not define whether its ten-year window indexes predictor years or realized outcome years. I apply it to the realized outcome formation labels, `t−10` through `t−1`, so all ten observations are prior to the current formation label. This is a best-effort reading of “previous ten years”; it is not the latest date-available window. Some March-`t` statements can be public by June `t`, but the paper calls those current-year inputs and does not say to include current-year outcomes in estimating coefficients. No alternative window is run.

| June coefficient origin | Horizon | Observed predictor formation labels | Observed outcome formation labels | Predictor FYEAR | Predictor DATADATE | Outcome FYEAR | Outcome DATADATE |
|---|---:|---:|---:|---:|---|---:|---|
| 1968 | 1 | 1964–1966 | 1965–1967 | 1963–1965 | 30 Jun 1963–31 Mar 1966 | 1964–1966 | 30 Jun 1964–31 Mar 1967 |
| 1968 | 3 | 1964 only | 1967 only | 1963 only | 30 Jun 1963–31 Mar 1964 | 1966 only | 30 Jun 1966–31 Mar 1967 |
| 2008 | 1 | 1997–2006 | 1998–2007 | 1995–2005 | 30 Apr 1996–31 Mar 2006 | 1996–2006 | 30 Apr 1997–31 Mar 2007 |
| 2008 | 3 | 1995–2004 | 1998–2007 | 1993–2003 | 30 Apr 1994–31 Mar 2004 | 1996–2006 | 30 Apr 1997–31 Mar 2007 |

The formation label is `year(DATADATE)+(month(DATADATE)>=4)`, matching the paper's April-prior-year through March-current-year June-information window (physical p. 5 and Fig. 1; printed p. 508). These are the actual complete-case boundary values from the SAS fit-pool diagnostic, not calendar-year approximations. At the 1968 origin, only three outcome formation years are available for horizon 1 and one for horizon 3 because the local annual file's first accepted formation year is 1964; the first-year complete-case pools are 346 and 45 rows, respectively. At the 2008 origin, all ten outcome formation years are present for both horizons. Thus the early regressions are full rank but do not have ten years of outcome history. The local `CRSP.MSF` itself spans 31 Dec 1925–31 Dec 2024 (5,153,763 rows); the paper's July 1963–June 2009 dates describe its study intersection, not the local file's overall start. The build uses the June security rows from 1963 through 2008, with 2,087 securities in June 1963.

## Source fields and construction

The exact source tables read are `COMP.FUNDA`, `COMP.G_EXRT_DLY`, `COMP.G_EXRT_MTH`, `CRSP.CCMXPF_LINKTABLE`, `CRSP.DSENAMES`, and `CRSP.MSF`. The annual accounting product is filtered to `INDL` / `C` / `D` / `STD`; the universal `INDL` product is retained for all industries, including financials and utilities. `IB`, `AT`, `ACT`, `CHE`, `LCT`, `DLC`, `TXP`, `DP`, and `OANCF` use the exact catalog mnemonics. The paper describes earnings as income before extraordinary items; it does not prescribe a different Compustat item.

`D_t` is mapped to `DVC`, the provider's Cash Dividends—Common item A21, defined as common-stock dividends declared in the year and measured in millions (S&P Research Insight North America Data Guide, physical p. 292 / printed p. 262). This is the closest documented dollar-level common-equity dividend measure. The paper does not print a Compustat item number, so this mapping remains best effort. An observed `DVC=0` is a nonpayer zero; `DVC` missing remains missing. `DD=1` only when observed `DVC>0`, and `DD=0` when observed `DVC≤0`. `NegE=1` when observed raw `IB<0`, `0` when observed `IB≥0`, and missing when `IB` is missing.

Before fiscal year 1988, accruals are `(ACT−CHE)` change minus `(LCT−DLC−TXP)` change minus `DP`, using the exact preceding fiscal-year record for the same GVKEY. From FY1988 onward, `AC=IB−OANCF`, irrespective of whether cash flow fields happen to be present before 1988 (paper physical p. 3 and Table 1 note; printed pp. 506, 509). Any required current/prior balance or current CFO item missing makes accruals unavailable; none is zero-filled. The method switch uses Compustat `FYEAR`, while the regression-year alignment uses the actual `DATADATE` publication-origin mapping.

Compustat's period-specific `CURCD` identifies the reporting currency. Provider amounts are in millions, and the paper reports all level variables in $ millions (Table 1 note, physical p. 6 / printed p. 509). The build translates monetary inputs to USD with the catalog-listed daily and monthly exchange-rate tables. The provider identifies GBP as the primary currency and gives the cross-rate formula: reported amount ÷ GBP-to-reporting rate × GBP-to-USD rate. Balance-sheet amounts (`AT`, `ACT`, `CHE`, `LCT`, `DLC`, `TXP`) use daily `EXRATD` at the statement date; income/cash-flow amounts (`IB`, `OANCF`, `DP`) and dividends use the 12-month average `EXRAT12M` for that reporting month. This follows the provider's currency process and timing table (Xpressfeed *Using the Data*, physical pp. 82–89, especially physical p. 87 / printed p. 79). USD rows use the identity conversion. For pre-1988 accruals, each current and prior balance sheet is converted at its own statement-date rates, and current `DP` uses its own 12-month flow rate, before taking the dollar differences. A non-USD value is unavailable when its currency is missing or an exact dated rate leg is absent, nonpositive, or ambiguous. The build retains and reports both FX legs and daily rate-type provenance; it does not drop quotes solely because `EXRATTPD` is missing or differs between legs.

Local FX coverage was verified on the exact joins. `G_EXRT_DLY` has GBP quote dates from 1 Feb 1982 to 31 Mar 2008, including 9,148 unique positive GBP-to-USD dates; `G_EXRT_MTH` runs from 31 Jan 1982 to 31 Mar 2008, with 315 unique positive GBP-to-USD month dates. The annual statement pool contains 33,410 CAD rows and 338,431 USD rows. USD uses identity rates. Of CAD rows, 25,331 have both unique positive daily legs and 25,903 have both monthly legs. The match audit found no ambiguous or nonpositive quote pairs. Missing conversions are due to absent exact-date legs: none of the CAD rows before the local 1982 rate start converts; in 1982, 18/288 rows have daily rates and 31/288 have monthly rates; monthly quotes match every row from 1983 onward, while daily exact-date matches rise from 432/462 in 1983, 134/521 in 1984, 473/558 in 1985, 538/595 in 1986, to all 639/639 in 1987 and complete coverage thereafter. The reason audit retains this missingness rather than shifting the sample start or requiring equal `EXRATTPD` types. Only 303 converted CAD rows enter the full historical training pools, alongside 3,552,070 USD rows.

The security population follows the paper's historical CRSP share codes 10/11 and NYSE/Amex/Nasdaq exchange codes 1/2/3 at the June formation date. The CRSP guide defines first-digit 1 as ordinary common shares and second-digit 8 as REITs, while second-digit 4 denotes closed-end funds; first-digit 3 denotes ADRs (physical PDF p. 115 / printed p. 111). Thus the direct 10/11 rule excludes those coded security types. The build audits the overlap with Compustat historical SIC 6798 and does not add an SIC restriction to the stated CRSP share-code screen.

`GVKEY` is linked to the dated security identifier `LPERMNO` at June 30 of the formation year, then screened using the `DSENAMES` interval effective on that date and the June row in `MSF`. Only `LINKTYPE` LC/LU and dated `LINKPRIM` P/C records are accepted as a best-effort standard/primary link rule. CRSP documents P as Compustat primary and C as the CRSP-assigned primary used to resolve overlapping or absent primary markers; LC is research-complete and LU is an unresearched CUSIP link (CRSP/Compustat Merged Database Guide, physical pp. 8–9 and 21–22). Company fundamentals remain at GVKEY grain and the retained market identifier is security-level PERMNO, not PERMCO. Current/header issue identifiers are not used as historical evidence. CCM effective dates are tested at the June formation date, with missing starts treated as open and all missing end codes—including local `.E`—treated as open. The build reports open-end and ambiguous-link counts. Where a GVKEY maps to more than one qualifying PERMNO at a formation date, that company-year is excluded rather than duplicated or arbitrarily assigned.

The local CCM file has 92,711 links, including 14,750 `.E` open end dates, all handled as open. No GVKEY-year mapped to multiple qualifying PERMNOs; no PERMNO-year mapped to multiple GVKEYs; and no effective June name interval had conflicting share/exchange codes. The latest-dated-row tie checks for GVKEY/FYEAR and GVKEY/formation-year each found zero tied groups.

For each formation year and each of `IB`, `AT`, `DVC`, and `AC`, the build computes 1st/99th percentiles on the USD-translated CRSP/Compustat primary-security cross-section and winsorizes values to those limits. Dummies remain binary and are not winsorized. Target earnings use the cutoff for their own formation year. This implements the paper's annual winsorization statement (physical p. 4; printed p. 507) without pooling years or using cutoffs from a later year. Financial firms and utilities remain in the baseline.

## Estimation and inference

Stata's installed `xtfmb` package estimates the horizon-specific pooled OLS cross-sections by fit year and averages the annual coefficients equally. Newey–West inference uses lag 3 as an explicit best-effort choice because the paper does not report a bandwidth. The package's lagged covariance has zero off-diagonal elements; only the individual coefficient standard errors/t statistics are interpreted, and no joint HAC test is reported. The `xtfmb` `R-squared` is the mean annual R-squared. Native annual OLS fit-statistic checks supply the mean adjusted R-squared reported in Table 1. The template's p-value presentation is used; the source reports t-statistics but does not state the exact p-value tail convention.

The initial Stata export used a nonstandard `e(ar2)` scalar, leaving the adjusted R-squared row blank. I corrected the stored scalar to `e(r2_a)` and reran only the Stata cell; the estimates and SAS handoff did not change. This is the final export.

The data have one record for each qualifying security training observation, horizon, outcome formation year, and rolling June fit year. `panelid` identifies that repeated underlying training observation as it enters successive rolling windows. It satisfies the package's panel/time declaration only; the Newey–West ordering is the annual fit-year coefficient sequence, not a concatenation of firm observations. Full-rank checks report N and design rank for all 41 year/horizon fits before the average table is written. In Table 1 export, `Observations` is the sum across overlapping rolling training fits, not a unique-firm count; per-origin Ns are in the SAS and Stata logs.

## Interpretation limits

Published Table 1 Panel B benchmarks are `E_t` slopes 0.8304, 0.7924, and 0.7871; `AC_t` slopes −0.0725, −0.0848, and −0.0932; and adjusted R-squared 0.86, 0.81, and 0.78 (physical p. 6 / printed p. 509). These are comparison points only and are not used to choose the screens, data mappings, or lag. The exact rolling-fit Ns, early-window convention, dividend item, primary-link policy, and Newey–West lag are not reported. Current local source vintages, dated FX-rate gaps or conflicts, primary-link ambiguity, and any remaining missing fields can produce differences from the paper.

For Table 1A, the source's Panel A is the time-series average of annual cross-sectional statistics over 1968–2008 (physical p. 6 / printed p. 509). The predecessor SAS listing's broad descriptive diagnostic averages 45 formation years (1964–2008), including four warm-up years, so it is not used as the Table 1A comparison. A bounded A2 audit recomputes the same annual cross-sectional moments and then equally averages the 41 study years; those comparison values and the unresolved differences are recorded in `review.md`.

The executed `xtfmb` estimates and comparisons are:

| Horizon | Total rolling-fit rows | Annual fit periods | Earnings slope (t) | Accrual slope (t) | Mean adjusted R-squared | Published earnings / accrual / adjusted R-squared |
|---|---:|---:|---:|---:|---:|---|
| 1 year | 1,304,760 | 41 | 0.7841 (28.59) | −0.0488 (−3.87) | 0.8543 | 0.8304 / −0.0725 / 0.86 |
| 2 years | 1,181,563 | 41 | 0.7175 (20.76) | −0.0485 (−2.42) | 0.8004 | 0.7924 / −0.0848 / 0.81 |
| 3 years | 1,066,050 | 41 | 0.7225 (14.22) | −0.0455 (−1.93) | 0.7658 | 0.7871 / −0.0932 / 0.78 |

The listed observation counts sum across overlapping ten-year rolling fits, not unique firm-years. The t-statistics use lag 3 and the package's individual-coefficient Newey–West variances. The fit periods are all full rank, including the shortened 1968 warmup described above. These estimate differences are reported as observed; they were not used to revise the population, winsorization, currency mapping, or window.


In [ ]:
options errorabend msglevel=i fullstimer;

libname comp "/storage/sas/base/comp/d_na" access=readonly;
libname fx "/storage/sas/base/comp/d_global/currency" access=readonly;
libname crsp ("/storage/sas/base/crsp/a_ccm" "/storage/sas/base/crsp/a_stock") access=readonly;
libname out "./data";

/* Keep the source-period accounting history needed for each current and prior
   fiscal-year balance. The file is filtered before any cross-table joins. */
proc sql;
    create table work.fraw0 as
    select gvkey, datadate, fyear, curcd, indfmt, consol, popsrc, datafmt,
           at, ib, dvc, act, che, lct, dlc, txp, dp, oancf, sich
    from comp.funda
    where datadate between '01JAN1957'd and '31MAR2008'd
      and not missing(gvkey)
      and not missing(datadate)
      and not missing(fyear)
      and indfmt = 'INDL'
      and consol = 'C'
      and popsrc = 'D'
      and datafmt = 'STD';
quit;

data work.fraw1;
    set work.fraw0;
    formyr = year(datadate) + (month(datadate) >= 4);
    if 1958 <= formyr and formyr <= 2008;
run;

proc sql;
    create table work.fraw as select distinct * from work.fraw1;

    create table work.fymax as
    select gvkey, fyear, max(datadate) as maxdate format=date9.
    from work.fraw
    group by gvkey, fyear;

    create table work.fytop as
    select a.*
    from work.fraw as a inner join work.fymax as b
      on a.gvkey = b.gvkey and a.fyear = b.fyear and a.datadate = b.maxdate;

    create table work.fytopcnt as
    select gvkey, fyear, count(*) as ntop
    from work.fytop
    group by gvkey, fyear;

    create table work.fyearone as
    select a.*
    from work.fytop as a inner join work.fytopcnt as b
      on a.gvkey = b.gvkey and a.fyear = b.fyear
    where b.ntop = 1;

    select count(*) as tied_latest_fyear_groups
    from work.fytopcnt where ntop > 1;
quit;

/* The provider's cross-rate formula uses GBP as its primary currency:
   reported amount / GBP-to-reporting rate * GBP-to-target rate. Keep all
   quote-type descriptors; rate type is provenance, not an eligibility rule. */
data work.fxday0;
    set fx.g_exrt_dly(keep=datadate fromcurd tocurd exratd exrattpd);
    where datadate >= '01JAN1957'd and datadate <= '31MAR2008'd
      and fromcurd = 'GBP';
run;

data work.fxmon0;
    set fx.g_exrt_mth(keep=datadate fromcurm tocurm exrat12m);
    where datadate >= '01JAN1957'd and datadate <= '31MAR2008'd
      and fromcurm = 'GBP';
run;

proc sql;
    create table work.fxdayg as
    select datadate, tocurd, min(exratd) as fxmin, max(exratd) as fxmax,
           count(distinct exratd) as nrates,
           count(distinct exrattpd) as ntypes,
           min(exrattpd) as typemin, max(exrattpd) as typemax
    from work.fxday0
    where not missing(tocurd)
    group by datadate, tocurd;

    create table work.fxmong as
    select datadate, tocurm, min(exrat12m) as fxmin,
           max(exrat12m) as fxmax, count(distinct exrat12m) as nrates
    from work.fxmon0
    where not missing(tocurm)
    group by datadate, tocurm;
quit;

data work.fxday;
    set work.fxdayg;
    fxrate = .;
    if nrates = 1 and fxmin > 0 and fxmin = fxmax then fxrate = fxmin;
run;

data work.fxmon;
    set work.fxmong;
    fxrate = .;
    if nrates = 1 and fxmin > 0 and fxmin = fxmax then fxrate = fxmin;
run;

proc sql;
    select min(datadate) as dly_first format=date9.,
           max(datadate) as dly_last format=date9., count(*) as dly_gbp_rows
    from work.fxday0;
    select min(datadate) as mth_first format=date9.,
           max(datadate) as mth_last format=date9., count(*) as mth_gbp_rows
    from work.fxmon0;
    select min(datadate) as dly_usd_first format=date9.,
           max(datadate) as dly_usd_last format=date9.,
           count(*) as dly_usd_quote_dates
    from work.fxday where tocurd = 'USD' and not missing(fxrate);
    select min(datadate) as mth_usd_first format=date9.,
           max(datadate) as mth_usd_last format=date9.,
           count(*) as mth_usd_quote_dates
    from work.fxmon where tocurm = 'USD' and not missing(fxrate);
quit;

/* Exact-date joins keep the balance sheet at its report-date rate and flow
   items at the 12-month average for that same reported period. */
proc sql;
    create table work.fxstat as
    select a.*,
           dr.fxrate as drep, dr.nrates as drep_nrate,
           dr.ntypes as drep_ntype, dr.typemin as drep_typemin,
           dr.typemax as drep_typemax,
           du.fxrate as dusd, du.nrates as dusd_nrate,
           du.ntypes as dusd_ntype, du.typemin as dusd_typemin,
           du.typemax as dusd_typemax,
           mr.fxrate as mrep, mr.nrates as mrep_nrate,
           mu.fxrate as musd, mu.nrates as musd_nrate
    from work.fyearone as a
    left join work.fxday as dr
      on a.datadate = dr.datadate and a.curcd = dr.tocurd
    left join work.fxday as du
      on a.datadate = du.datadate and du.tocurd = 'USD'
    left join work.fxmon as mr
      on a.datadate = mr.datadate and a.curcd = mr.tocurm
    left join work.fxmon as mu
      on a.datadate = mu.datadate and mu.tocurm = 'USD';
quit;

data work.fxstat1;
    set work.fxstat;
    d_factor = .;
    m_factor = .;
    if curcd = 'USD' then do;
        d_factor = 1;
        m_factor = 1;
    end;
    else if curcd = 'GBP' then do;
        if not missing(dusd) and dusd > 0 then d_factor = dusd;
        if not missing(musd) and musd > 0 then m_factor = musd;
    end;
    else if not missing(curcd) then do;
        if not missing(drep) and not missing(dusd) and drep > 0 and dusd > 0 then
            d_factor = dusd / drep;
        if not missing(mrep) and not missing(musd) and mrep > 0 and musd > 0 then
            m_factor = musd / mrep;
    end;

    at_usd = .; act_usd = .; che_usd = .; lct_usd = .;
    dlc_usd = .; txp_usd = .;
    ib_usd = .; dvc_usd = .; dp_usd = .; ocf_usd = .;
    if not missing(at) and not missing(d_factor) then at_usd = at * d_factor;
    if not missing(act) and not missing(d_factor) then act_usd = act * d_factor;
    if not missing(che) and not missing(d_factor) then che_usd = che * d_factor;
    if not missing(lct) and not missing(d_factor) then lct_usd = lct * d_factor;
    if not missing(dlc) and not missing(d_factor) then dlc_usd = dlc * d_factor;
    if not missing(txp) and not missing(d_factor) then txp_usd = txp * d_factor;
    if not missing(ib) and not missing(m_factor) then ib_usd = ib * m_factor;
    if not missing(dvc) and not missing(m_factor) then dvc_usd = dvc * m_factor;
    if not missing(dp) and not missing(m_factor) then dp_usd = dp * m_factor;
    if not missing(oancf) and not missing(m_factor) then ocf_usd = oancf * m_factor;
run;

/* Prior balance-sheet amounts are already translated at their own dates. */
proc sql;
    create table work.accr0 as
    select a.*,
           b.datadate as pdate format=date9., b.curcd as pcurcd,
           b.act_usd as pact_usd, b.che_usd as pche_usd,
           b.lct_usd as plct_usd, b.dlc_usd as pdlc_usd,
           b.txp_usd as ptxp_usd
    from work.fxstat1 as a
    left join work.fxstat1 as b
      on a.gvkey = b.gvkey and b.fyear = a.fyear - 1
    where 1962 <= a.formyr and a.formyr <= 2008;
quit;

data work.accr1;
    set work.accr0;
    accr = .;
    if fyear < 1988 then do;
        if nmiss(act_usd, che_usd, lct_usd, dlc_usd, txp_usd, dp_usd,
                 pact_usd, pche_usd, plct_usd, pdlc_usd, ptxp_usd) = 0 then
            accr = (act_usd - che_usd) - (pact_usd - pche_usd)
                   - ((lct_usd - dlc_usd - txp_usd)
                      - (plct_usd - pdlc_usd - ptxp_usd)) - dp_usd;
    end;
    else if fyear >= 1988 and nmiss(ib_usd, ocf_usd) = 0 then
        accr = ib_usd - ocf_usd;
    if fyear >= 1963 and 1963 <= formyr and formyr <= 2008;
run;

proc sql;
    create table work.formmax as
    select gvkey, formyr, max(datadate) as maxdate format=date9.
    from work.accr1
    group by gvkey, formyr;

    create table work.formtop as
    select a.*
    from work.accr1 as a inner join work.formmax as b
      on a.gvkey = b.gvkey and a.formyr = b.formyr and a.datadate = b.maxdate;

    create table work.formcnt as
    select gvkey, formyr, count(*) as ntop
    from work.formtop
    group by gvkey, formyr;

    create table work.ann as
    select a.*
    from work.formtop as a inner join work.formcnt as b
      on a.gvkey = b.gvkey and a.formyr = b.formyr
    where b.ntop = 1;

    select count(*) as tied_latest_formation_groups
    from work.formcnt where ntop > 1;

    select formyr, min(datadate) as first_statement format=date9.,
           max(datadate) as last_statement format=date9., count(*) as funda_rows
    from work.ann group by formyr order by formyr;

    select curcd, count(*) as annual_rows,
           sum(not missing(d_factor)) as daily_factor_rows,
           sum(not missing(m_factor)) as monthly_factor_rows,
           sum(not missing(at_usd)) as at_usd_rows,
           sum(not missing(ib_usd)) as ib_usd_rows,
           sum(not missing(dvc_usd)) as dvc_usd_rows,
           sum(not missing(accr)) as accr_rows
    from work.ann group by curcd order by curcd;

    select formyr, curcd, count(*) as annual_rows,
           sum(not missing(d_factor)) as daily_factor_rows,
           sum(not missing(m_factor)) as monthly_factor_rows,
           sum(not missing(at_usd)) as at_usd_rows,
           sum(not missing(ib_usd)) as ib_usd_rows,
           sum(not missing(dvc_usd)) as dvc_usd_rows,
           sum(not missing(accr)) as accr_rows
    from work.ann group by formyr, curcd order by formyr, curcd;

    /* For non-USD currencies, separate an absent dated leg from a date with
       an empty, nonpositive, or conflicting quote pair. */
    select formyr, curcd, count(*) as annual_rows,
           sum(missing(drep_nrate)) as drep_absent,
           sum(drep_nrate = 0) as drep_no_numeric_quote,
           sum(drep_nrate > 1) as drep_ambiguous,
           sum(drep_nrate = 1 and missing(drep)) as drep_nonpositive,
           sum(not missing(drep)) as drep_unique_positive,
           sum(missing(dusd_nrate)) as dusd_absent,
           sum(dusd_nrate = 0) as dusd_no_numeric_quote,
           sum(dusd_nrate > 1) as dusd_ambiguous,
           sum(dusd_nrate = 1 and missing(dusd)) as dusd_nonpositive,
           sum(not missing(dusd)) as dusd_unique_positive,
           sum(missing(mrep_nrate)) as mrep_absent,
           sum(mrep_nrate = 0) as mrep_no_numeric_quote,
           sum(mrep_nrate > 1) as mrep_ambiguous,
           sum(mrep_nrate = 1 and missing(mrep)) as mrep_nonpositive,
           sum(not missing(mrep)) as mrep_unique_positive,
           sum(missing(musd_nrate)) as musd_absent,
           sum(musd_nrate = 0) as musd_no_numeric_quote,
           sum(musd_nrate > 1) as musd_ambiguous,
           sum(musd_nrate = 1 and missing(musd)) as musd_nonpositive,
           sum(not missing(musd)) as musd_unique_positive,
           sum(not missing(d_factor)) as daily_conversion_rows,
           sum(not missing(m_factor)) as monthly_conversion_rows
    from work.ann
    where not missing(curcd) and curcd not in ('USD', 'GBP')
    group by formyr, curcd order by formyr, curcd;
quit;

proc sql;
    select count(*) as ccm_link_rows,
           sum(missing(linkdt)) as open_or_missing_link_starts,
           sum(missing(linkenddt)) as open_or_missing_link_ends,
           sum(linkenddt = .E) as ccm_end_e_rows
    from crsp.ccmxpf_linktable;
quit;

proc sql;
    create table work.linkcand as
    select distinct a.gvkey, a.formyr, a.datadate, a.fyear, a.curcd,
           l.lpermno as permno
    from work.ann as a
    inner join crsp.ccmxpf_linktable as l
      on a.gvkey = l.gvkey
     and l.linktype in ('LC', 'LU')
     and l.linkprim in ('P', 'C')
     and (missing(l.linkdt) or l.linkdt <= mdy(6, 30, a.formyr))
     and (missing(l.linkenddt) or l.linkenddt >= mdy(6, 30, a.formyr))
    where not missing(l.lpermno) and l.lpermno ne 0;

    create table work.linkcnt as
    select gvkey, formyr, count(*) as nperm
    from work.linkcand group by gvkey, formyr;

    create table work.linkone as
    select a.*
    from work.linkcand as a inner join work.linkcnt as b
      on a.gvkey = b.gvkey and a.formyr = b.formyr
    where b.nperm = 1;

    create table work.permgv as
    select distinct permno, formyr, gvkey from work.linkone;

    create table work.permcnt as
    select permno, formyr, count(*) as ngv
    from work.permgv group by permno, formyr;

    create table work.linkuniq as
    select a.*
    from work.linkone as a inner join work.permcnt as b
      on a.permno = b.permno and a.formyr = b.formyr
    where b.ngv = 1;

    select count(*) as gvkeyyr_multi_permno from work.linkcnt where nperm > 1;
    select count(*) as permyr_multi_gvkey from work.permcnt where ngv > 1;
quit;

proc sql;
    create table work.namecand as
    select distinct l.permno, l.formyr, n.shrcd, n.exchcd
    from work.linkuniq as l inner join crsp.dsenames as n
      on l.permno = n.permno
     and (missing(n.namedt) or n.namedt <= mdy(6, 30, l.formyr))
     and (missing(n.nameendt) or n.nameendt >= mdy(6, 30, l.formyr));

    create table work.namecnt as
    select permno, formyr, count(*) as ncode
    from work.namecand group by permno, formyr;

    create table work.nameone as
    select a.*
    from work.namecand as a inner join work.namecnt as b
      on a.permno = b.permno and a.formyr = b.formyr
    where b.ncode = 1 and a.shrcd in (10, 11) and a.exchcd in (1, 2, 3);

    select count(*) as permyr_ambig_name from work.namecnt where ncode > 1;
quit;

proc sql;
    select min(date) as first_msf_date format=date9.,
           max(date) as last_msf_date format=date9.,
           count(*) as msf_rows
    from crsp.msf;
quit;

data work.junemsf;
    set crsp.msf(keep=permno date);
    where date >= '01JUN1963'd and date <= '30JUN2008'd;
    if month(date) = 6;
    formyr = year(date);
    keep permno formyr date;
run;

proc sql;
    create table work.junesec as
    select distinct permno, formyr from work.junemsf;

    select formyr, count(*) as june_securities
    from work.junesec group by formyr order by formyr;
quit;

proc sql;
    create table work.pop0 as
    select distinct a.gvkey, l.permno, a.formyr, a.datadate, a.fyear,
           a.curcd, a.sich, a.d_factor, a.m_factor,
           a.drep, a.drep_ntype, a.drep_typemin, a.drep_typemax,
           a.dusd, a.dusd_ntype, a.dusd_typemin, a.dusd_typemax,
           a.mrep, a.musd, a.ib_usd, a.at_usd, a.dvc_usd, a.accr,
           n.shrcd, n.exchcd
    from work.ann as a
    inner join work.linkuniq as l
      on a.gvkey = l.gvkey and a.formyr = l.formyr
    inner join work.nameone as n
      on l.permno = n.permno and l.formyr = n.formyr
    inner join work.junesec as j
      on n.permno = j.permno and n.formyr = j.formyr
    where n.shrcd in (10, 11) and n.exchcd in (1, 2, 3);

    select formyr, count(*) as primary_share_n,
           count(distinct gvkey) as gvkeys,
           sum(curcd = 'USD') as usd_rows,
           sum(curcd ne 'USD' and not missing(curcd)) as nonusd_rows,
           sum(missing(curcd)) as missing_currency_rows,
           sum(not missing(at_usd)) as n_at_usd,
           sum(not missing(ib_usd)) as n_ib_usd,
           sum(not missing(dvc_usd)) as n_dvc_usd,
           sum(not missing(accr)) as n_accr,
           sum(sich = 6798) as historical_sic_6798_rows
    from work.pop0 group by formyr order by formyr;

    select curcd, count(*) as eligible_rows,
           sum(not missing(d_factor)) as daily_factor_rows,
           sum(not missing(m_factor)) as monthly_factor_rows,
           sum(not missing(at_usd)) as n_at_usd,
           sum(not missing(ib_usd)) as n_ib_usd,
           sum(not missing(dvc_usd)) as n_dvc_usd,
           sum(not missing(accr)) as n_accr
    from work.pop0 group by curcd order by curcd;

    select sum(sich = 6798) as kept_sic_6798_rows from work.pop0;

    select formyr, sum(missing(dvc_usd)) as dvc_usd_missing,
           sum(dvc_usd = 0) as dvc_observed_zero,
           sum(dvc_usd > 0) as dvc_positive,
           sum(dvc_usd < 0) as dvc_negative
    from work.pop0 group by formyr order by formyr;
quit;

proc sort data=work.pop0 out=work.pop0s;
    by formyr;
run;

proc univariate data=work.pop0s noprint;
    by formyr;
    var ib_usd at_usd dvc_usd accr;
    output out=work.cuts
        pctlpts=1 99
        pctlpre=ib_ at_ dvc_ ac_
        pctlname=p1 p99;
run;

proc sql;
    create table work.anncut as
    select a.*, c.ib_p1, c.ib_p99, c.at_p1, c.at_p99,
           c.dvc_p1, c.dvc_p99, c.ac_p1, c.ac_p99
    from work.ann as a left join work.cuts as c
      on a.formyr = c.formyr;
quit;

data work.annw;
    set work.anncut;
    ibw = .; atw = .; dvw = .; acw = .;
    dd = .; negib = .;
    if not missing(ib) then negib = (ib < 0);
    if not missing(dvc) then dd = (dvc > 0);
    if not missing(ib_usd) and not missing(ib_p1) and not missing(ib_p99) then
        ibw = min(max(ib_usd, ib_p1), ib_p99);
    if not missing(at_usd) and not missing(at_p1) and not missing(at_p99) then
        atw = min(max(at_usd, at_p1), at_p99);
    if not missing(dvc_usd) and not missing(dvc_p1) and not missing(dvc_p99) then
        dvw = min(max(dvc_usd, dvc_p1), dvc_p99);
    if not missing(accr) and not missing(ac_p1) and not missing(ac_p99) then
        acw = min(max(accr, ac_p1), ac_p99);
    keep gvkey datadate fyear formyr curcd sich
         ibw atw dvw dd negib acw d_factor m_factor
         drep drep_ntype drep_typemin drep_typemax
         dusd dusd_ntype dusd_typemin dusd_typemax mrep musd;
run;

proc sql;
    create table work.popw as
    select a.gvkey, a.permno, a.formyr, a.datadate, a.fyear,
           a.curcd, a.sich, a.shrcd, a.exchcd,
           a.d_factor, a.m_factor,
           a.drep, a.drep_ntype, a.drep_typemin, a.drep_typemax,
           a.dusd, a.dusd_ntype, a.dusd_typemin, a.dusd_typemax,
           a.mrep, a.musd,
           b.ibw, b.atw, b.dvw, b.dd, b.negib, b.acw
    from work.pop0 as a inner join work.annw as b
      on a.gvkey = b.gvkey and a.formyr = b.formyr;

    select formyr, count(*) as eligible_security_rows,
           sum(curcd = 'USD') as usd_rows,
           sum(curcd ne 'USD' and not missing(curcd)) as nonusd_rows,
           sum(not missing(ibw)) as n_ibw,
           sum(not missing(atw)) as n_atw,
           sum(not missing(dvw)) as n_dvw,
           sum(not missing(dd)) as n_dd,
           sum(not missing(negib)) as n_negib,
           sum(not missing(acw)) as n_acw
    from work.popw group by formyr order by formyr;
quit;

proc sort data=work.popw out=work.popws;
    by formyr;
run;

proc means data=work.popws noprint;
    by formyr;
    var ibw atw dvw dd negib acw;
    output out=work.annstats(drop=_type_ _freq_)
        n=n_ib n_at n_dv n_dd n_nege n_ac
        mean=m_ib m_at m_dv m_dd m_nege m_ac
        p1=p1_ib p1_at p1_dv p1_dd p1_nege p1_ac
        p25=p25_ib p25_at p25_dv p25_dd p25_nege p25_ac
        median=med_ib med_at med_dv med_dd med_nege med_ac
        p75=p75_ib p75_at p75_dv p75_dd p75_nege p75_ac
        p99=p99_ib p99_at p99_dv p99_dd p99_nege p99_ac
        std=sd_ib sd_at sd_dv sd_dd sd_nege sd_ac;
run;

title "Annual-average cross-sectional summaries after USD translation and winsorization";
proc means data=work.annstats mean;
    var m_ib p1_ib p25_ib med_ib p75_ib p99_ib sd_ib
        m_at p1_at p25_at med_at p75_at p99_at sd_at
        m_dv p1_dv p25_dv med_dv p75_dv p99_dv sd_dv
        m_dd p1_dd p25_dd med_dd p75_dd p99_dd sd_dd
        m_nege p1_nege p25_nege med_nege p75_nege p99_nege sd_nege
        m_ac p1_ac p25_ac med_ac p75_ac p99_ac sd_ac;
run;
title;

/* For each historical predictor year, retain the realized Compustat earnings
   outcome at exactly tau years ahead. X is screened at the predictor year's
   CRSP/Compustat intersection; outcome earnings need not retain a future
   listing after delisting. Both amounts have their own annual USD cutoffs. */
proc sql;
    create table work.pairs0 as
    select x.gvkey, x.permno, x.formyr as xyear, x.datadate as xdate format=date9.,
           x.fyear as xfyear, x.curcd as xcur, x.shrcd, x.exchcd,
           x.d_factor as xd_factor, x.m_factor as xm_factor,
           x.drep as xdrep, x.drep_ntype as xdrep_ntype,
           x.drep_typemin as xdrep_typemin, x.drep_typemax as xdrep_typemax,
           x.dusd as xdusd, x.dusd_ntype as xdusd_ntype,
           x.dusd_typemin as xdusd_typemin, x.dusd_typemax as xdusd_typemax,
           x.mrep as xmrep, x.musd as xmusd,
           y.formyr as outyr, y.datadate as ydate format=date9.,
           y.fyear as yfyear, y.curcd as ycur, y.m_factor as ym_factor,
           y.mrep as ymrep, y.musd as ymusd, y.ibw as yib,
           x.atw, x.dvw, x.dd, x.ibw, x.negib, x.acw,
           y.formyr - x.formyr as tau
    from work.popw as x inner join work.annw as y
      on x.gvkey = y.gvkey
     and y.formyr between x.formyr + 1 and x.formyr + 3
    where 1963 <= x.formyr and x.formyr <= 2007
      and 1964 <= y.formyr and y.formyr <= 2007
      and not missing(x.atw)
      and not missing(x.dvw)
      and not missing(x.dd)
      and not missing(x.ibw)
      and not missing(x.negib)
      and not missing(x.acw)
      and not missing(y.ibw);
quit;

data work.train0;
    set work.pairs0;
    if tau not in (1, 2, 3) then delete;
    if xyear ne outyr - tau then delete;
    if ydate > mdy(3, 31, outyr) then delete;
    if xdate > mdy(3, 31, xyear) then delete;
    firstfit = max(1968, outyr + 1);
    lastfit = min(2008, outyr + 10);
    do fit_year = firstfit to lastfit;
        if fit_year - 10 <= outyr and outyr <= fit_year - 1 then output;
    end;
    drop firstfit lastfit;
run;

proc sort data=work.train0 out=work.trains;
    by tau gvkey permno outyr fit_year;
run;

data work.train;
    set work.trains;
    by tau gvkey permno outyr fit_year;
    retain panelid 0;
    if first.outyr then panelid + 1;
    keep fit_year tau gvkey permno outyr xyear xfyear yfyear xdate ydate
         xcur ycur shrcd exchcd panelid yib atw dvw dd ibw negib acw
         xd_factor xm_factor xdrep xdrep_ntype xdrep_typemin xdrep_typemax
         xdusd xdusd_ntype xdusd_typemin xdusd_typemax xmrep xmusd
         ym_factor ymrep ymusd;
run;

/* PROC REG is used only for annual fit diagnostics. ODS retains ANOVA model
   degrees of freedom and fit statistics; no parameter-estimate table is
   captured, printed, or retained. Stata xtfmb estimates average coefficients
   and HAC inference. */
proc sort data=work.train out=work.regin;
    by tau fit_year;
run;

ods select none;
ods output ANOVA=work.ods_anova FitStatistics=work.ods_fit;
proc reg data=work.regin;
    by tau fit_year;
    model yib = atw dvw dd ibw negib acw;
run;
quit;
ods output close;
ods select all;

proc sql;
    create table work.poolcounts as
    select tau, fit_year, count(*) as fit_n
    from work.train group by tau, fit_year;

    create table work.regrank as
    select tau, fit_year, max(df) as slope_rank
    from work.ods_anova
    where upcase(source) = 'MODEL'
    group by tau, fit_year;

    create table work.regfit as
    select tau, fit_year,
           max(case when label2 = 'R-Square' then nvalue2 else . end) as rsq,
           max(case when label2 = 'Adj R-Sq' then nvalue2 else . end) as ar2
    from work.ods_fit group by tau, fit_year;

    create table work.regstat as
    select a.tau, a.fit_year, a.fit_n, b.slope_rank, c.rsq, c.ar2
    from work.poolcounts as a
    left join work.regrank as b
      on a.tau = b.tau and a.fit_year = b.fit_year
    left join work.regfit as c
      on a.tau = c.tau and a.fit_year = c.fit_year;
quit;

proc datasets library=work nolist;
    delete regin ods_anova ods_fit;
quit;

data work.expected;
    do tau = 1 to 3;
        do fit_year = 1968 to 2008;
            output;
        end;
    end;
run;

proc sql;
    create table work.fitcheck as
    select e.tau, e.fit_year, coalesce(r.fit_n, 0) as fit_n,
           r.slope_rank, r.rsq, r.ar2,
           (not missing(r.slope_rank) and r.slope_rank = 6 and r.fit_n > 7) as fitok
    from work.expected as e left join work.regstat as r
      on e.tau = r.tau and e.fit_year = r.fit_year;

    select tau, fit_year, fit_n, slope_rank, rsq, ar2, fitok
    from work.fitcheck order by tau, fit_year;

    select tau, sum(fitok) as full_rank_origins,
           mean(case when fitok = 1 then ar2 else . end) as mean_adjusted_r2
    from work.fitcheck group by tau order by tau;

    create table work.fitmeans as
    select tau, sum(fitok) as full_rank_origins,
           mean(case when fitok = 1 then ar2 else . end) as mean_ar2
    from work.fitcheck group by tau;

    create table work.train2 as
    select a.*, b.fitok, c.full_rank_origins, c.mean_ar2 as ar2mean
    from work.train as a
    left join work.fitcheck as b
      on a.tau = b.tau and a.fit_year = b.fit_year
    left join work.fitmeans as c
      on a.tau = c.tau;

    select a.tau, a.fit_year, count(*) as complete_pool_n,
           max(a.fitok) as fitok, max(a.full_rank_origins) as full_rank_origins,
           max(a.ar2mean) as mean_adjusted_r2
    from work.train2 as a group by a.tau, a.fit_year order by a.tau, a.fit_year;
quit;

proc sql;
    create table work.keydups as
    select fit_year, tau, gvkey, permno, outyr, count(*) as nrows
    from work.train2
    group by fit_year, tau, gvkey, permno, outyr
    having calculated nrows > 1;

    select count(*) as duplicate_key_groups from work.keydups;

    select tau, fit_year, count(*) as complete_pool_n,
           min(xdate) as min_predictor_date format=date9.,
           max(xdate) as max_predictor_date format=date9.,
           min(ydate) as min_outcome_date format=date9.,
           max(ydate) as max_outcome_date format=date9.
    from work.train2 group by tau, fit_year order by tau, fit_year;

    select tau, fit_year,
           min(xyear) as first_predictor_formation,
           max(xyear) as last_predictor_formation,
           min(xfyear) as first_predictor_fyear,
           max(xfyear) as last_predictor_fyear,
           min(xdate) as first_predictor_date format=date9.,
           max(xdate) as last_predictor_date format=date9.,
           min(outyr) as first_outcome_formation,
           max(outyr) as last_outcome_formation,
           min(yfyear) as first_outcome_fyear,
           max(yfyear) as last_outcome_fyear,
           min(ydate) as first_outcome_date format=date9.,
           max(ydate) as last_outcome_date format=date9.
    from work.train2
    where fit_year in (1968, 2008) and tau in (1, 3)
    group by tau, fit_year order by tau, fit_year;

    select xcur as predictor_currency, count(*) as training_rows
    from work.train2 group by xcur order by xcur;

    select count(*) as future_outcome_date_violations
    from work.train2
    where outyr > fit_year - 1
       or ydate > mdy(3, 31, fit_year - 1)
       or xyear ne outyr - tau;
quit;

data _null_;
    if 0 then set work.keydups nobs=ndup;
    if ndup > 0 then do;
        put 'ERROR: Duplicate training-sample keys remain after source and link audits.';
        abort cancel;
    end;
    stop;
run;

data out.icc_firm_year;
    set work.train2;
    label fit_year = 'June forecast-origin year for rolling coefficient fit'
          tau = 'Earnings forecast horizon'
          outyr = 'Formation year of realized training earnings'
          xyear = 'Formation year of predictor record'
          yib = 'Winsorized future income before extraordinary items, USD millions'
          atw = 'Winsorized total assets, USD millions'
          dvw = 'Winsorized common dividends DVC, USD millions'
          dd = 'Indicator for observed common dividends greater than zero'
          ibw = 'Winsorized current income before extraordinary items, USD millions'
          negib = 'Indicator for negative current earnings'
          acw = 'Winsorized accruals, USD millions'
          fitok = 'Indicator that annual pooled OLS is full rank'
          ar2mean = 'Mean adjusted R-squared across full-rank annual fits'
          xcur = 'Predictor statement reporting currency'
          ycur = 'Outcome statement reporting currency'
          xd_factor = 'Daily conversion factor from reporting currency to USD'
          xm_factor = 'Monthly conversion factor from reporting currency to USD'
          ym_factor = 'Outcome monthly conversion factor from reporting currency to USD';
    format xdate ydate date9.;
run;

proc contents data=out.icc_firm_year;
run;

proc means data=work.train2 n nmiss mean p1 p50 p99;
    class tau;
    var yib atw dvw dd ibw negib acw;
run;
